In [ ]:
#@title Setup (run me first!) { display-mode: "form" }
#@markdown This cell loads the shared infrastructure, generates data, and
#@markdown precomputes all interactive elements. Takes ~20 seconds.

import json, os, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from statsmodels.sandbox.regression.gmm import IV2SLS
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
warnings.filterwarnings('ignore')

# ── Color palette ──
COLORS = {
    'ols': '#2E5EA8',
    'truth': '#D4A017',
    'bias': '#C0392B',
    'repair': '#27AE60',
    'alt': '#7F8C8D',
}

# ── Gate System ──
_trap_responses = {}
_TRAP_LOG_PATH = '/content/trap_log.json'

def _load_trap_log():
    global _trap_responses
    if os.path.exists(_TRAP_LOG_PATH):
        try:
            with open(_TRAP_LOG_PATH, 'r') as f:
                _trap_responses = json.load(f)
        except (json.JSONDecodeError, IOError):
            _trap_responses = {}

def _save_trap_log():
    try:
        os.makedirs(os.path.dirname(_TRAP_LOG_PATH), exist_ok=True)
        with open(_TRAP_LOG_PATH, 'w') as f:
            json.dump(_trap_responses, f, indent=2)
    except IOError:
        pass

def record_trap_response(notebook_id, question_id, response):
    key = f"{notebook_id}_{question_id}"
    _trap_responses[key] = {"response": response, "timestamp": datetime.now().isoformat()}
    _save_trap_log()

def get_trap_response(notebook_id, question_id):
    key = f"{notebook_id}_{question_id}"
    entry = _trap_responses.get(key)
    return entry["response"] if entry else None

def check_gate(notebook_id, question_id):
    return f"{notebook_id}_{question_id}" in _trap_responses

def get_all_responses():
    return dict(_trap_responses)

def create_trap_widget(notebook_id, question_id, question_text, options):
    label = widgets.HTML(f"<b>{question_text}</b>")
    radio = widgets.RadioButtons(options=options, value=None, layout=widgets.Layout(width='auto'))
    submit = widgets.Button(description="Submit", button_style='primary')
    output = widgets.Output()
    def on_submit(btn):
        with output:
            output.clear_output()
            if radio.value is None:
                print("Please select an option before submitting.")
                return
            record_trap_response(notebook_id, question_id, radio.value)
            print(f"Response recorded: {radio.value}")
            submit.disabled = True
            radio.disabled = True
    submit.on_click(on_submit)
    existing = get_trap_response(notebook_id, question_id)
    if existing is not None:
        try:
            radio.value = existing
            radio.disabled = True
            submit.disabled = True
        except Exception:
            pass
    return widgets.VBox([label, radio, submit, output])

_load_trap_log()

# ── CompanySimulator ──
class CompanySimulator:
    def __init__(self, endogeneity_strength=20, heteroscedasticity_strength=0.5,
                 nonlinearity=True, bad_control_strength=0.1, noise_sigma=1.0):
        self.endogeneity_strength = endogeneity_strength
        self.heteroscedasticity_strength = heteroscedasticity_strength
        self.nonlinearity = nonlinearity
        self.bad_control_strength = bad_control_strength
        self.noise_sigma = noise_sigma
        self.beta_0 = 50
        self.beta_1 = 8
        self.beta_2 = 3
        self.beta_U = 2
        self.staff_loading = 5

    def generate(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        eta1 = rng.normal(0, self.noise_sigma, n)
        eta2 = rng.normal(0, self.noise_sigma, n)
        eta3 = rng.normal(0, self.noise_sigma, n)
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity:
            X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2
        eps_std = np.sqrt(np.abs(self.heteroscedasticity_strength * X1))
        eps = rng.normal(0, eps_std)
        if self.nonlinearity:
            Y = self.beta_0 + self.beta_1 * np.log(X1) + self.beta_2 * X2 + self.beta_U * U + eps
        else:
            Y = self.beta_0 + self.beta_1 * X1 + self.beta_2 * X2 + self.beta_U * U + eps
        X3 = self.bad_control_strength * Y + eta3
        return pd.DataFrame({'revenue': Y, 'ad_spend': X1, 'staff_count': X2, 'satisfaction': X3})

    def truth(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        eta1 = rng.normal(0, self.noise_sigma, n)
        eta2 = rng.normal(0, self.noise_sigma, n)
        eta3 = rng.normal(0, self.noise_sigma, n)
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity:
            X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2
        eps_std = np.sqrt(np.abs(self.heteroscedasticity_strength * X1))
        eps = rng.normal(0, eps_std)
        if self.nonlinearity:
            Y = self.beta_0 + self.beta_1 * np.log(X1) + self.beta_2 * X2 + self.beta_U * U + eps
        else:
            Y = self.beta_0 + self.beta_1 * X1 + self.beta_2 * X2 + self.beta_U * U + eps
        X3 = self.bad_control_strength * Y + eta3
        df = pd.DataFrame({'revenue': Y, 'ad_spend': X1, 'staff_count': X2,
                           'satisfaction': X3, 'demand_U': U})
        params = {'beta_0': self.beta_0, 'beta_1': self.beta_1, 'beta_2': self.beta_2,
                  'beta_U': self.beta_U, 'sigma_epsilon': self.heteroscedasticity_strength}
        return df, params

    def dgp_summary(self):
        func = r"\log(X_1)" if self.nonlinearity else r"X_1"
        return (rf"$Y = {self.beta_0} + {self.beta_1} \cdot {func} "
                rf"+ {self.beta_2} \cdot X_2 + {self.beta_U} \cdot U + \varepsilon$")

    def set_heteroscedasticity(self, strength):
        self.heteroscedasticity_strength = strength
    def set_endogeneity(self, strength):
        self.endogeneity_strength = strength
    def set_nonlinearity(self, curvature):
        self.nonlinearity = bool(curvature)

# ── MonteCarloEngine ──
class MonteCarloEngine:
    def run(self, estimator_fn, param_name, param_grid, simulator, n_reps=5000, n_obs=500):
        results = np.empty((len(param_grid), n_reps))
        try:
            progress = widgets.IntProgress(value=0, min=0, max=len(param_grid),
                                           description='Simulating:', style={'description_width': 'initial'})
            display(progress)
            use_widget = True
        except Exception:
            use_widget = False
        setter = getattr(simulator, f'set_{param_name}', None)
        for i, val in enumerate(param_grid):
            if setter is not None:
                setter(val)
            for r in range(n_reps):
                data = simulator.generate(n=n_obs, seed=r)
                results[i, r] = estimator_fn(data)
            if use_widget:
                progress.value = i + 1
        if use_widget:
            progress.bar_style = 'success'
        return results

    def quick_run(self, estimator_fn, dgp_fn, n_reps=1000, n_obs=500):
        results = np.empty(n_reps)
        for r in range(n_reps):
            data = dgp_fn(seed=r, n=n_obs)
            results[r] = estimator_fn(data)
        return results

# ── AutopsyReport ──
class AutopsyReport:
    @staticmethod
    def lightweight(notebook_number):
        threat = widgets.Text(description='Biggest threat:', placeholder='What is the biggest threat to this estimate?',
                              layout=widgets.Layout(width='90%'), style={'description_width': '120px'})
        check = widgets.Text(description='How to check:', placeholder='How would you check if that threat is real?',
                             layout=widgets.Layout(width='90%'), style={'description_width': '120px'})
        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()
        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                record_trap_response(nb_id, "threat", threat.value)
                record_trap_response(nb_id, "check", check.value)
                print("Autopsy responses saved.")
                submit.disabled = True
        submit.on_click(on_submit)
        return widgets.VBox([widgets.HTML(f"<h3>Mini Autopsy \u2014 Notebook {notebook_number}</h3>"),
                             threat, check, submit, output])

# Fast mode: set FAST_TEST=1 to reduce precomputation
FAST_TEST = os.environ.get('FAST_TEST', '0') == '1'

# ── Generate data ──
sim = CompanySimulator()
df = sim.generate(n=500, seed=42)
df_truth, true_params = sim.truth(n=500, seed=42)

# Fit the "best" model from previous notebooks: log(ad_spend) + staff_count
X_good = pd.DataFrame({
    'const': 1,
    'log_ad_spend': np.log(df['ad_spend']),
    'staff_count': df['staff_count']
}, index=df.index)
model_good = sm.OLS(df['revenue'], X_good).fit(use_t=True)
ols_coef = model_good.params['log_ad_spend']

# ── Precompute endogeneity slider cache ──
print("Precomputing endogeneity simulations...")
endog_strengths = np.array([0, 2, 5, 10, 15, 20, 25, 30, 40, 50])
n_reps_endog = 50 if FAST_TEST else 500

progress = widgets.IntProgress(value=0, min=0, max=len(endog_strengths),
                               description='Simulating:', style={'description_width': 'initial'})
display(progress)

endog_cache = {}
true_causal = sim.beta_1  # True causal effect of log(ad_spend) on revenue

for i, strength in enumerate(endog_strengths):
    ols_ests = []
    exp_ests = []
    for r in range(n_reps_endog):
        rng = np.random.default_rng(r)
        n = 500
        U = rng.standard_normal(n)
        eta1 = rng.normal(0, 1, n)
        eta2 = rng.normal(0, 1, n)

        # Observational: X1 driven by U
        X1 = np.abs(strength * U + eta1) + 0.01
        X2 = 5 * U + eta2
        eps = rng.normal(0, np.sqrt(np.abs(0.5 * X1)))
        Y = 50 + 8 * np.log(X1) + 3 * X2 + 2 * U + eps

        X_obs = sm.add_constant(np.log(X1))
        m_obs = sm.OLS(Y, X_obs).fit()
        ols_ests.append(m_obs.params[1])

        # Experimental: X1 randomly assigned
        X1_exp = rng.uniform(X1.min(), X1.max(), n)
        eps_exp = rng.normal(0, np.sqrt(np.abs(0.5 * X1_exp)))
        Y_exp = 50 + 8 * np.log(X1_exp) + 3 * X2 + 2 * U + eps_exp

        X_exp = sm.add_constant(np.log(X1_exp))
        m_exp = sm.OLS(Y_exp, X_exp).fit()
        exp_ests.append(m_exp.params[1])

    endog_cache[strength] = {
        'ols_mean': np.mean(ols_ests),
        'ols_std': np.std(ols_ests),
        'exp_mean': np.mean(exp_ests),
        'exp_std': np.std(exp_ests),
        'ols_ests': np.array(ols_ests),
        'exp_ests': np.array(exp_ests),
        'gap': np.mean(ols_ests) - np.mean(exp_ests),
    }
    progress.value = i + 1

# ── Precompute IV strength cache ──
print("Precomputing IV strength simulations...")
iv_strengths = np.array([0.5, 1, 2, 5, 10, 15, 20, 30, 50, 100])
n_reps_iv = 50 if FAST_TEST else 500

progress2 = widgets.IntProgress(value=0, min=0, max=len(iv_strengths),
                                description='IV sims:', style={'description_width': 'initial'})
display(progress2)

iv_cache = {}
for i, iv_str in enumerate(iv_strengths):
    iv_ests = []
    iv_cis = []
    f_stats = []
    for r in range(n_reps_iv):
        rng = np.random.default_rng(r + 10000)
        n = 500
        U = rng.standard_normal(n)
        Z = rng.standard_normal(n)  # Instrument: random ad price variation
        eta1 = rng.normal(0, 1, n)
        eta2 = rng.normal(0, 1, n)

        # X1 driven by both U (endogeneity) and Z (instrument)
        X1 = np.abs(20 * U + iv_str * Z + eta1) + 0.01
        log_X1 = np.log(X1)
        X2 = 5 * U + eta2
        eps = rng.normal(0, np.sqrt(np.abs(0.5 * X1)))
        Y = 50 + 8 * log_X1 + 3 * X2 + 2 * U + eps

        # First stage: log(X1) on Z
        Z_mat = sm.add_constant(Z)
        first_stage = sm.OLS(log_X1, Z_mat).fit()
        f_stat = first_stage.fvalue
        f_stats.append(f_stat)

        # Manual 2SLS
        log_X1_hat = first_stage.predict(Z_mat)
        X_second = sm.add_constant(log_X1_hat)
        second_stage = sm.OLS(Y, X_second).fit()
        iv_ests.append(second_stage.params[1])
        iv_cis.append(second_stage.conf_int().iloc[1].values)

    iv_ests_arr = np.array(iv_ests)
    iv_cis_arr = np.array(iv_cis)
    ci_widths = iv_cis_arr[:, 1] - iv_cis_arr[:, 0]

    iv_cache[iv_str] = {
        'iv_mean': np.mean(iv_ests_arr),
        'iv_std': np.std(iv_ests_arr),
        'iv_ests': iv_ests_arr,
        'mean_f': np.mean(f_stats),
        'median_ci_width': np.median(ci_widths),
        'f_stats': np.array(f_stats),
    }
    progress2.value = i + 1

progress.bar_style = 'success'
progress2.bar_style = 'success'
print("Setup complete!")
print(f"Data: {len(df)} observations. OLS coefficient on log(ad_spend): {ols_coef:.2f}")
print(f"True causal effect: {true_causal}")

# Notebook 6: Why Your Causal Claim Is Wrong

*If your input and your output cause each other, regression can't tell you which direction the effect runs.*

---

You've done everything right. You controlled for confounders (Notebook 1), used robust standard errors (Notebook 2), distinguished significance from importance (Notebook 3), specified the right functional form (Notebook 4), and avoided overfitting (Notebook 5).

Your model looks solid. Let's check.

In [ ]:
# ── The Setup: Everything looks solid ──

print("Your best model so far:")
print("  - Log transformation (Notebook 4)")
print("  - Staff count as control")
print("  - Robust standard errors")
print()

# Fit with robust SEs
model_robust = sm.OLS(df['revenue'], X_good).fit(cov_type='HC1')
print(model_robust.summary())
print(f"\nCoefficient on log(ad_spend): {model_robust.params['log_ad_spend']:.3f}")
print(f"95% CI: [{model_robust.conf_int().loc['log_ad_spend'][0]:.3f}, "
      f"{model_robust.conf_int().loc['log_ad_spend'][1]:.3f}]")
print(f"p-value: {model_robust.pvalues['log_ad_spend']:.6f}")
print(f"R-squared: {model_robust.rsquared:.3f}")

In [ ]:
# ── The Trap ──
# The coefficient implies: a 1% increase in ad_spend raises revenue by ~coef/100

implied_effect = model_robust.params['log_ad_spend'] * np.log(1 + 100/df['ad_spend'].mean())

trap_widget = create_trap_widget(
    'notebook_6', 'causal_claim',
    f'Does increasing ad spend by $100K CAUSE revenue to increase by '
    f'~${implied_effect*1000:,.0f}?',
    [
        'Yes \u2014 the coefficient is significant with robust SEs and correct functional form',
        'No \u2014 there must be reverse causality or another issue',
        "Can't determine from this regression \u2014 correlation doesn't prove causation",
    ]
)
display(trap_widget)

In [ ]:
# ── The Reveal: Reverse causality exposed ──

if not check_gate('notebook_6', 'causal_claim'):
    print("\u26a0\ufe0f Please submit your answer in the cell above before continuing.")
else:
    student_answer = get_trap_response('notebook_6', 'causal_claim')

    # Show the feedback loop: revenue LEADS ad spend
    # Simulate time series to show the feedback loop
    rng_ts = np.random.default_rng(42)
    T = 60  # months
    demand = np.zeros(T)
    ad_spend_ts = np.zeros(T)
    revenue_ts = np.zeros(T)

    demand[0] = rng_ts.standard_normal()
    for t in range(T):
        demand[t] = 0.8 * demand[max(0, t-1)] + rng_ts.standard_normal()
        # Revenue driven by demand
        revenue_ts[t] = 50 + 2 * demand[t] + rng_ts.normal(0, 0.5)
        # Ad spend driven by LAST PERIOD's revenue (feedback)
        ad_spend_ts[t] = max(1, 10 + 0.3 * revenue_ts[max(0, t-1)] + rng_ts.normal(0, 1))

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))

    # Top: Time series showing revenue leads ad spend
    ax = axes[0]
    # Normalize for visual comparison
    rev_norm = (revenue_ts - revenue_ts.mean()) / revenue_ts.std()
    ad_norm = (ad_spend_ts - ad_spend_ts.mean()) / ad_spend_ts.std()

    months = np.arange(T)
    ax.plot(months, rev_norm, color=COLORS['truth'], linewidth=2.5,
            linestyle='-', marker='D', markevery=5, markersize=5,
            label='Revenue (normalized)')
    ax.plot(months, ad_norm, color=COLORS['ols'], linewidth=2,
            linestyle='-', marker='o', markevery=5, markersize=5,
            label='Ad Spend (normalized)')

    # Highlight lead-lag with arrows at a few points
    for t in [10, 25, 40]:
        if rev_norm[t] > ad_norm[t]:
            ax.annotate('', xy=(t+1, ad_norm[t+1]), xytext=(t, rev_norm[t]),
                        arrowprops=dict(arrowstyle='->', color=COLORS['bias'],
                                        lw=2, alpha=0.6))

    ax.set_xlabel('Month', fontsize=12)
    ax.set_ylabel('Normalized Value', fontsize=12)
    ax.set_title('Revenue LEADS Ad Spend \u2014 The Company Increases Ads AFTER High Revenue',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)

    # Bottom: Cross-correlation
    ax = axes[1]
    max_lag = 12
    lags = range(-max_lag, max_lag + 1)
    xcorr = [np.corrcoef(revenue_ts[max_lag:-max_lag],
                         np.roll(ad_spend_ts, -lag)[max_lag:-max_lag])[0, 1]
             for lag in lags]
    colors_bar = [COLORS['truth'] if l < 0 else COLORS['ols'] for l in lags]
    ax.bar(lags, xcorr, color=colors_bar, alpha=0.7)
    ax.axvline(0, color=COLORS['alt'], linestyle='--', alpha=0.5)
    ax.set_xlabel('Lag (months)', fontsize=12)
    ax.set_ylabel('Cross-correlation', fontsize=12)
    ax.set_title('Cross-correlation: Revenue at time t vs. Ad Spend at time t+lag\n'
                 'Gold bars (negative lag) = revenue leads; Blue bars (positive lag) = ad spend leads',
                 fontsize=11)

    plt.tight_layout()
    plt.show()

    print("\nThe feedback loop:")
    print("  1. High demand \u2192 high revenue")
    print("  2. High revenue \u2192 company increases ad budget")
    print("  3. OLS sees ad_spend and revenue move together")
    print("  4. OLS captures the CORRELATION, not the CAUSAL direction")
    print(f"\nOLS coefficient ({ols_coef:.2f}) overstates the true causal effect ({true_causal}).")
    print("Part of what OLS captures is reverse causality: revenue \u2192 ad spend.")

    if 'Yes' in student_answer:
        print("\n\u274c You said 'Yes'. The model looked solid, but it can't distinguish")
        print("causation from correlation when there's a feedback loop.")
    elif "Can't" in student_answer:
        print("\n\u2714\ufe0f Correct! Regression alone cannot establish causation.")
    else:
        print("\n\u2714\ufe0f Good instinct! There IS reverse causality here.")

## The Problem: Endogeneity

When $E[\varepsilon | X] \neq 0$, OLS is **inconsistent** — it converges to the wrong value no matter how much data you collect.

In our data, unobserved demand $U$ drives both ad spend and revenue. The regression captures this confounding as part of the "effect" of ad spend. No amount of correct standard errors, functional form, or cross-validation can fix this.

$$\hat{\beta}_{OLS} \xrightarrow{p} \beta + \frac{\text{Cov}(X, \varepsilon)}{\text{Var}(X)} \neq \beta$$

The bias doesn't shrink with more data. It's baked in.

In [ ]:
# ── The Destruction Playground: Endogeneity strength slider ──
# Simulate A/B test vs. observational. Gap widens with endogeneity.

endog_slider = widgets.SelectionSlider(
    options=[(f'{int(v)}', v) for v in endog_strengths],
    value=endog_strengths[0],
    description='Endogeneity:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='80%'),
    continuous_update=False
)

destroy_output = widgets.Output()
destroy_summary = widgets.Output()

def update_endog(change):
    strength = change['new']
    c = endog_cache[strength]

    with destroy_output:
        destroy_output.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5))

        # Histogram of OLS vs experimental estimates
        bins = np.linspace(min(c['exp_ests'].min(), c['ols_ests'].min()) - 1,
                          max(c['exp_ests'].max(), c['ols_ests'].max()) + 1, 40)

        ax.hist(c['ols_ests'], bins=bins, alpha=0.6, color=COLORS['ols'],
                edgecolor='white', label=f'Observational OLS (mean={c["ols_mean"]:.2f})')
        ax.hist(c['exp_ests'], bins=bins, alpha=0.6, color=COLORS['repair'],
                edgecolor='white', label=f'Experimental/A|B (mean={c["exp_mean"]:.2f})')
        ax.axvline(true_causal, color=COLORS['truth'], linewidth=3,
                   linestyle='-', label=f'True causal effect = {true_causal}')

        status = 'No endogeneity' if strength == 0 else f'Endogeneity strength = {int(strength)}'
        ax.set_title(f'{status} | OLS bias = {c["ols_mean"] - true_causal:+.2f} | '
                     f'OLS-Experimental gap = {c["gap"]:+.2f}',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Estimated coefficient on log(ad_spend)', fontsize=11)
        ax.set_ylabel('Frequency', fontsize=11)
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

    with destroy_summary:
        destroy_summary.clear_output(wait=True)
        print(f"Endogeneity: {int(strength)}")
        print(f"True causal:   {true_causal:.2f}")
        print(f"OLS mean:      {c['ols_mean']:.2f}")
        print(f"Experiment:    {c['exp_mean']:.2f}")
        print(f"OLS bias:      {c['ols_mean'] - true_causal:+.2f}")
        print(f"OLS-Exp gap:   {c['gap']:+.2f}")
        if strength == 0:
            print("\nNo confounding: OLS = experiment")
        elif strength >= 20:
            print("\nStrong confounding: OLS is")
            print("seriously misleading.")
            print("Only randomization recovers truth.")

endog_slider.observe(update_endog, names='value')
update_endog({'new': endog_strengths[0]})

layout = widgets.HBox([
    widgets.VBox([endog_slider, destroy_output], layout=widgets.Layout(width='75%')),
    destroy_summary
])
display(layout)

### Discussion

> *A study finds that countries with more UN peacekeepers have more violence. Does peacekeeping cause violence?*

Think about what's driving the correlation before continuing.

## The Repair: Instrumental Variables

When you can't run an experiment and you can't observe the confounder, you need an **instrument**.

An instrument $Z$ must satisfy two conditions:
1. **Relevance**: $Z$ is correlated with $X$ (i.e., $\text{Cov}(Z, X) \neq 0$)
2. **Exclusion**: $Z$ affects $Y$ only through $X$ (i.e., $\text{Cov}(Z, \varepsilon) = 0$)

In our NovaMart story: **ad prices varied randomly across regions** due to media market fluctuations. Regions where ads were cheaper bought more ads — but the price variation had nothing to do with local demand.

This was foreshadowed in Notebook 1: when you can't observe the confounder and can't control for it, you need an instrument.

In [ ]:
# ── The Repair: IV in 3 steps ──

# Step 1: What's an instrument?
print("=" * 60)
print("INSTRUMENTAL VARIABLES IN THREE STEPS")
print("=" * 60)

# Generate IV data
rng_iv = np.random.default_rng(42)
n = 500
U = rng_iv.standard_normal(n)
Z = rng_iv.standard_normal(n)  # Instrument: random ad price variation
eta1 = rng_iv.normal(0, 1, n)
X1 = np.abs(20 * U + 20 * Z + eta1) + 0.01  # Strong instrument
log_X1 = np.log(X1)
X2 = 5 * U + rng_iv.normal(0, 1, n)
eps = rng_iv.normal(0, np.sqrt(np.abs(0.5 * X1)))
Y = 50 + 8 * log_X1 + 3 * X2 + 2 * U + eps

print("\nStep 1: The Instrument")
print("  Z = random variation in ad prices across regions")
print(f"  Corr(Z, log_X1) = {np.corrcoef(Z, log_X1)[0,1]:.3f}  (relevance \u2713)")
print(f"  Corr(Z, U)      = {np.corrcoef(Z, U)[0,1]:.3f}  (exclusion \u2713)")

print("\n" + "-" * 60)
print("Step 2: First Stage \u2014 Predict X from Z")
Z_mat = sm.add_constant(Z)
first_stage = sm.OLS(log_X1, Z_mat).fit()
print(f"  log(ad_spend) = {first_stage.params[0]:.3f} + {first_stage.params[1]:.3f} * Z")
print(f"  F-statistic: {first_stage.fvalue:.1f}  (rule of thumb: F > 10)")
print(f"  R\u00b2: {first_stage.rsquared:.3f}")

print("\n" + "-" * 60)
print("Step 3: Second Stage \u2014 Regress Y on predicted X")
log_X1_hat = first_stage.predict(Z_mat)
X_second = sm.add_constant(log_X1_hat)
second_stage = sm.OLS(Y, X_second).fit()

# Also OLS for comparison
X_ols = sm.add_constant(log_X1)
ols_comparison = sm.OLS(Y, X_ols).fit()

print(f"\n  OLS estimate: {ols_comparison.params[1]:.3f}  (biased by confounding)")
print(f"  IV estimate:  {second_stage.params[1]:.3f}  (uses only exogenous variation)")
print(f"  True value:   {true_causal:.3f}")
print(f"\n  IV recovers the truth: bias = {second_stage.params[1] - true_causal:+.3f}")
print(f"  OLS was biased by:              {ols_comparison.params[1] - true_causal:+.3f}")

In [ ]:
# ── Visualization: OLS vs IV ──

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: OLS (contaminated by U)
scatter_colors = plt.cm.RdYlGn_r((U - U.min()) / (U.max() - U.min()))
ax1.scatter(log_X1, Y, c=U, cmap='RdYlGn_r', alpha=0.5, s=15)
x_line = np.linspace(log_X1.min(), log_X1.max(), 100)
ax1.plot(x_line, ols_comparison.params[0] + ols_comparison.params[1] * x_line,
         color=COLORS['ols'], linewidth=2.5, linestyle='-',
         label=f'OLS: {ols_comparison.params[1]:.2f}')
ax1.axhline(0, color=COLORS['alt'], linestyle=':', alpha=0.3)
ax1.set_xlabel('log(Ad Spend)', fontsize=11)
ax1.set_ylabel('Revenue', fontsize=11)
ax1.set_title(f'OLS = {ols_comparison.params[1]:.2f} (confounded by demand)',
              fontsize=12, fontweight='bold')
ax1.legend(fontsize=11)
cbar1 = plt.colorbar(ax1.collections[0], ax=ax1)
cbar1.set_label('Unobserved Demand (U)', fontsize=10)

# Right: IV (only uses Z-driven variation)
ax2.scatter(log_X1_hat, Y, c=Z, cmap='coolwarm', alpha=0.5, s=15)
x_line2 = np.linspace(log_X1_hat.min(), log_X1_hat.max(), 100)
ax2.plot(x_line2, second_stage.params[0] + second_stage.params[1] * x_line2,
         color=COLORS['repair'], linewidth=2.5, linestyle='-.',
         marker='s', markevery=15,
         label=f'IV: {second_stage.params[1]:.2f}')
ax2.set_xlabel('Predicted log(Ad Spend) from instrument', fontsize=11)
ax2.set_ylabel('Revenue', fontsize=11)
ax2.set_title(f'IV = {second_stage.params[1]:.2f} (true = {true_causal})',
              fontsize=12, fontweight='bold')
ax2.legend(fontsize=11)
cbar2 = plt.colorbar(ax2.collections[0], ax=ax2)
cbar2.set_label('Instrument Z (ad price variation)', fontsize=10)

fig.suptitle('OLS Uses All Variation (Including Confounded). IV Uses Only Clean Variation.',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── The Limit: Weak instruments ──
# Slider: instrument strength (first-stage F). Weak instrument = chaos.

iv_slider = widgets.SelectionSlider(
    options=[(f'F\u2248{iv_cache[v]["mean_f"]:.0f}', v) for v in iv_strengths],
    value=iv_strengths[0],
    description='Instrument strength:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='80%'),
    continuous_update=False
)

iv_output = widgets.Output()
iv_summary = widgets.Output()

def update_iv(change):
    strength = change['new']
    c = iv_cache[strength]

    with iv_output:
        iv_output.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5))

        # Plot IV estimate distribution
        ests = c['iv_ests']
        # Clip extreme values for visualization
        ests_clipped = np.clip(ests, -50, 60)
        bins = np.linspace(ests_clipped.min() - 1, ests_clipped.max() + 1,
                          min(50, len(np.unique(ests_clipped))))

        ax.hist(ests_clipped, bins=bins, alpha=0.7, color=COLORS['repair'],
                edgecolor='white',
                label=f'IV estimates (mean={c["iv_mean"]:.2f}, SD={c["iv_std"]:.2f})')
        ax.axvline(true_causal, color=COLORS['truth'], linewidth=3,
                   linestyle='-', label=f'True = {true_causal}')
        ax.axvline(c['iv_mean'], color=COLORS['bias'], linewidth=2,
                   linestyle=':', label=f'IV mean = {c["iv_mean"]:.2f}')

        f_status = 'STRONG' if c['mean_f'] > 10 else 'WEAK!'
        ax.set_title(f'Instrument strength = {strength:.0f} | Mean F = {c["mean_f"]:.1f} [{f_status}] | '
                     f'Median CI width = {c["median_ci_width"]:.1f}',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('IV Estimate', fontsize=11)
        ax.set_ylabel('Frequency', fontsize=11)
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

    with iv_summary:
        iv_summary.clear_output(wait=True)
        print(f"Instrument: {strength:.0f}")
        print(f"Mean F-stat: {c['mean_f']:.1f}")
        print(f"True causal: {true_causal}")
        print(f"IV mean:     {c['iv_mean']:.2f}")
        print(f"IV std:      {c['iv_std']:.2f}")
        print(f"CI width:    {c['median_ci_width']:.1f}")
        if c['mean_f'] < 10:
            print("\n\u26a0\ufe0f WEAK INSTRUMENT!")
            print("F < 10: IV estimates")
            print("are unreliable. Wild")
            print("oscillations, huge CIs.")
        elif c['mean_f'] > 50:
            print("\n\u2714\ufe0f Strong instrument.")
            print("IV is stable and")
            print("close to truth.")

iv_slider.observe(update_iv, names='value')
update_iv({'new': iv_strengths[0]})

layout = widgets.HBox([
    widgets.VBox([iv_slider, iv_output], layout=widgets.Layout(width='75%')),
    iv_summary
])
display(layout)

In [ ]:
#@title Real-World Disaster: Police and Crime { display-mode: "form" }

# ── Sidebar: Police and crime ──

story_tab = widgets.HTML(value="""
<div style='padding: 15px; font-size: 13px; line-height: 1.6;'>
<h4>The Story</h4>
<p>For decades, a puzzling empirical regularity haunted criminologists: cities with
more police officers had <b>more crime</b>. A naive regression of crime rates on police
force size produces a <b>positive</b> coefficient.</p>

<p>Does policing cause crime? Obviously not. The feedback loop is clear: cities with
more crime <b>hire more police</b>. The OLS estimate captures this reverse causality.</p>

<p>Steven Levitt (2002) found a clever instrument: <b>electoral cycles</b>. Mayors tend to
increase police hiring in election years to appear "tough on crime," regardless of
actual crime trends. This political motivation is correlated with police size (relevance)
but plausibly unrelated to crime (exclusion).</p>

<p>Using electoral cycles as an instrument, Levitt found that the true effect of police
on crime is <b>negative</b> \u2014 more police reduces crime. The OLS estimate had the wrong
sign entirely.</p>
</div>
""")

math_tab = widgets.HTML(value="""
<div style='padding: 15px; font-size: 13px; line-height: 1.6;'>
<h4>The Math: Feedback Loop DGP</h4>
<p>True model:</p>
<ul>
<li>Crime = \u03b1 + \u03b2 \u00b7 Police + \u03b5 &nbsp;&nbsp;(\u03b2 < 0: more police reduces crime)</li>
<li>Police = \u03b3 + \u03b4 \u00b7 Crime + \u03b7 &nbsp;&nbsp;(\u03b4 > 0: more crime leads to more police)</li>
</ul>

<p>OLS of Crime on Police yields:</p>
<p style='text-align: center; font-size: 16px;'>
$\\hat{\\beta}_{OLS} \\xrightarrow{p} \\beta + \\frac{\\delta \\cdot \\sigma^2_\\varepsilon}{\\sigma^2_{Police}}$
</p>

<p>Since \u03b4 > 0 (reverse causality) and \u03c3\u00b2 > 0, the bias is <b>positive</b>.
If the bias exceeds |\u03b2|, OLS gets the <b>wrong sign</b>.</p>

<p>IV using electoral cycle Z:</p>
<ul>
<li>Relevance: Corr(Z, Police) \u2260 0 (mayors hire more police in election years)</li>
<li>Exclusion: Corr(Z, \u03b5) = 0 (electoral timing doesn't directly cause crime)</li>
</ul>
</div>
""")

sim_tab = widgets.Output()

def run_police_sim(btn):
    with sim_tab:
        sim_tab.clear_output(wait=True)
        rng = np.random.default_rng(42)
        n = 500
        n_reps_police = 50 if FAST_TEST else 500

        ols_ests = []
        iv_ests = []

        for r in range(n_reps_police):
            rng_r = np.random.default_rng(r)
            # Exogenous shocks
            eps = rng_r.normal(0, 3, n)  # crime shock
            eta = rng_r.normal(0, 2, n)  # police budget shock
            Z = rng_r.binomial(1, 0.3, n)  # election year

            # Solve simultaneous equations
            # Crime = 100 - 2*Police + eps
            # Police = 20 + 0.5*Crime + 5*Z + eta
            # Substituting: Crime = 100 - 2*(20 + 0.5*Crime + 5*Z + eta) + eps
            # Crime = 100 - 40 - Crime - 10*Z - 2*eta + eps
            # 2*Crime = 60 - 10*Z - 2*eta + eps
            crime = (60 - 10*Z - 2*eta + eps) / 2
            police = 20 + 0.5*crime + 5*Z + eta

            # OLS
            X_ols = sm.add_constant(police)
            m_ols = sm.OLS(crime, X_ols).fit()
            ols_ests.append(m_ols.params[1])

            # IV using election year
            Z_mat = sm.add_constant(Z)
            first = sm.OLS(police, Z_mat).fit()
            police_hat = first.predict(Z_mat)
            X_iv = sm.add_constant(police_hat)
            m_iv = sm.OLS(crime, X_iv).fit()
            iv_ests.append(m_iv.params[1])

        ols_arr = np.array(ols_ests)
        iv_arr = np.array(iv_ests)

        fig, ax = plt.subplots(figsize=(10, 5))
        bins = np.linspace(-6, 4, 50)
        ax.hist(ols_arr, bins=bins, alpha=0.6, color=COLORS['ols'], edgecolor='white',
                label=f'OLS (mean={np.mean(ols_arr):.2f})')
        ax.hist(iv_arr, bins=bins, alpha=0.6, color=COLORS['repair'], edgecolor='white',
                label=f'IV (mean={np.mean(iv_arr):.2f})')
        ax.axvline(-2, color=COLORS['truth'], linewidth=3, linestyle='-',
                   label='True effect = -2')
        ax.axvline(0, color=COLORS['alt'], linestyle=':', alpha=0.5)

        ax.set_title('Police and Crime: OLS Gets the Wrong Sign, IV Gets It Right',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Estimated effect of police on crime', fontsize=11)
        ax.set_ylabel('Frequency', fontsize=11)
        ax.legend(fontsize=10)
        plt.tight_layout()
        plt.show()

        print(f"True effect:        -2.00 (more police reduces crime)")
        print(f"OLS mean estimate:  {np.mean(ols_arr):+.2f} (WRONG SIGN due to feedback)")
        print(f"IV mean estimate:   {np.mean(iv_arr):+.2f} (correct sign, close to truth)")
        print(f"\nOLS says police CAUSE crime. IV says police REDUCE crime.")
        print(f"The difference: IV strips out the reverse causality.")

run_btn = widgets.Button(description='Run Simulation', button_style='primary')
run_btn.on_click(run_police_sim)

sim_content = widgets.VBox([run_btn, sim_tab])

tabs = widgets.Tab(children=[story_tab, math_tab, sim_content])
tabs.set_title(0, 'The Story')
tabs.set_title(1, 'The Math')
tabs.set_title(2, 'The Simulation')

display(tabs)

In [ ]:
# ── Mini Autopsy ──
display(AutopsyReport.lightweight(6))

---

## Bridge

*You've learned six ways your regression can fail. Now let's see what happens when they all attack at once.*

In the real world, data doesn't come with labels telling you which pathology is present. Multiple problems coexist. Your job is to diagnose them — all of them — simultaneously.

**Next: Notebook 7 — The Real World**

In [ ]:
# ── Transition Card ──
display(HTML("""
<div style='background: linear-gradient(135deg, #1a1a2e, #16213e); padding: 30px;
            border-radius: 10px; margin: 20px 0; color: white; font-family: sans-serif;'>
    <p style='font-size: 14px; opacity: 0.8; margin: 0;'>NEXT IN THE SERIES</p>
    <h2 style='margin: 10px 0; font-size: 24px;'>Notebook 7: The Real World</h2>
    <p style='font-size: 16px; opacity: 0.9; margin: 10px 0;'>
        You can do everything right and still get the wrong answer. Because the problem
        isn't in your model. It's in your data.
    </p>
    <a href='07_real_world.ipynb' style='display: inline-block; background: #D4A017;
       color: #1a1a2e; padding: 10px 24px; border-radius: 5px; text-decoration: none;
       font-weight: bold; margin-top: 10px;'>Open Notebook 7 \u2192</a>
</div>
"""))